In [ ]:
import os

from typing import Tuple

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

# 0- Functions

In [ ]:
def evaluate_uncertainty(
    y_true: np.ndarray, y_pred: np.ndarray, total_unc: np.ndarray, num_bins: int = 50
) -> dict:
    """
    Evaluate uncertainty calibration and quality metrics.

    Parameters
    ----------
    y_true : np.ndarray
        Ground truth target values. Shape (N,).
    y_pred : np.ndarray
        Predicted mean values. Shape (N,).
    total_unc : np.ndarray
        Predicted total uncertainties (variances). Shape (N,).
    num_bins : int, optional
        Number of confidence levels to evaluate miscalibration area, by default 50.

    Returns
    -------
    dict
        Dictionary with:
        - 'NLL': Negative log-likelihood
        - 'MiscalibrationArea': Area between ideal and empirical coverage
        - 'SpearmanR': Spearman rank correlation between abs error and uncertainty
    """
    y_true, y_pred, total_unc = (
        np.asarray(y_true),
        np.asarray(y_pred),
        np.asarray(total_unc),
    )

    nll = 0.5 * np.mean(
        np.log(2 * np.pi * total_unc) + ((y_true - y_pred) ** 2) / total_unc
    )

    sigmas = np.sqrt(total_unc)
    abs_error = np.abs(y_true - y_pred)
    alpha_levels = np.linspace(0.05, 0.95, num_bins)
    z_scores = norm.ppf(0.5 + alpha_levels / 2)

    empirical_coverages = []
    for z in z_scores:
        inside = (abs_error <= z * sigmas).mean()
        empirical_coverages.append(inside)

    ideal_coverages = alpha_levels
    miscal_area = np.trapz(
        np.abs(np.array(empirical_coverages) - ideal_coverages), alpha_levels
    )

    rho, _ = spearmanr(abs_error, np.sqrt(total_unc))

    return {"NLL": nll, "MiscalibrationArea": miscal_area, "SpearmanR": rho}

# 1- Model

## 1-1- Results

### 1-1-1- General Prediction and UQ Performances

In [ ]:
total_fv_df = pd.read_csv("../feature_vectors/all_data.csv")

In [ ]:
root_dir = "./experiment_results/"

all_results = []

n_split = 1

for split in sorted(os.listdir(root_dir)):

    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        n_model = 1
        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)
            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_ffnn_model_test_set.csv")

                if os.path.exists(test_file):
                    df = pd.read_csv(test_file)

                    df["split"] = n_split
                    df["model"] = n_model

                    all_results.append(df)
                    n_model += 1

        n_split += 1

if all_results:
    aggregated_results = pd.concat(all_results, ignore_index=True)

    aggregated_results.to_csv(
        "./experiment_results/aggregated_test_results.csv", index=False
    )
    print("Aggregated results saved to 'aggregated_test_results.csv'")
else:
    print("No test results found!")

In [ ]:
root_dir = "./experiment_results"
all_results = []

n_split = 1

for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        n_model = 1
        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)
            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_mcdropout_test_set.csv")

                if os.path.exists(test_file):
                    unc_df = pd.read_csv(test_file)

                    metrics = evaluate_uncertainty(
                        y_true=unc_df["y_test"].to_numpy(),
                        y_pred=unc_df["y_pred"].to_numpy(),
                        total_unc=unc_df["total_unc"].to_numpy(),
                    )

                    all_results.append(
                        {
                            "split": n_split,
                            "model": n_model,
                            "NLL": metrics["NLL"],
                            "MCA": metrics["MiscalibrationArea"],
                            "SpearmanR": metrics["SpearmanR"],
                        }
                    )

                    n_model += 1

        n_split += 1

if all_results:
    aggregated_results = pd.DataFrame(all_results)
    output_path = "./experiment_results/unc_aggregated_test_metrics.csv"
    aggregated_results.to_csv(output_path, index=False)
    print(f"Aggregated results saved to '{output_path}'")
else:
    print("No test results found!")

In [ ]:
per_metrics_df = pd.read_csv("./experiment_results/aggregated_test_results.csv")
un_metrics_df = pd.read_csv("./experiment_results/unc_aggregated_test_metrics.csv")

In [ ]:
per_metrics_df.iloc[:, 1:].describe().iloc[1:3, :-2].round(3)

In [ ]:
un_metrics_df.iloc[:, 2:].describe().iloc[1:3, :].round(3)